# Chain Rule and Computation Graphs

| Difficulty | ████░░░░░░ 4/10 |
| :--- | :--- |
| Prerequisites | 02.03 |
| Time | ~70 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/02_Calculus/04_chain_rule_and_computation_graphs.ipynb)

---

## 🎯 Learning Objective

Represent computations as directed acyclic graphs and derive forward-mode and reverse-mode differentiation from the multivariate chain rule.

---

## 📐 Theory

This is the single most important notebook in Module 02: backpropagation, the algorithm that
trains every neural network, is nothing more than the chain rule applied systematically.

### The univariate chain rule, again

From `02.02`: if $y=f(u)$ and $u=g(x)$, then $\frac{dy}{dx} = \frac{dy}{du}\cdot\frac{du}{dx}$.
Composing functions multiplies their local rates of change. Think of every computation as a
**chain of simple steps**, each with an easy local derivative, and let the chain rule stitch
the local pieces into the global derivative.

### The multivariate chain rule

When an intermediate quantity feeds multiple downstream paths, each path contributes:

$$\frac{\partial z}{\partial x} = \sum_{i} \frac{\partial z}{\partial u_i}\cdot\frac{\partial u_i}{\partial x}$$

if $x$ influences $z$ through several intermediates $u_1,\dots,u_k$. This "sum over paths" is
exactly what makes the computation-graph picture below necessary — a variable used twice must
have its influence counted twice.

### Computation graphs

Any composite expression is a **directed acyclic graph (DAG)**: nodes are variables (inputs, or
operation outputs), edges point from inputs to the operation consuming them. $z=(x+y)\cdot x$
decomposes as $a=x+y$, $z=a\cdot x$ — a graph where $x$ feeds both the $+$ node and the
$\times$ node (a "fan-out"), precisely the case the sum-over-paths rule handles.

### Forward-mode vs. reverse-mode differentiation

**Forward mode** propagates derivatives alongside values, in the same order as the original
computation: for a fixed input $x$, compute $\partial(\text{node})/\partial x$ from its
parents' values. This gets the derivative of every output w.r.t. *one* input per pass.

**Reverse mode** (backpropagation) runs the other way: a full forward pass computes every
node's value, then a backward walk computes $\partial(\text{output})/\partial(\text{node})$
using the cached values. This gets the derivative of *one* output w.r.t. *every* input in a
single pass — exactly a loss function's shape (millions of parameters, one scalar loss). This
asymmetry is why every deep learning framework uses reverse mode: its cost is roughly one extra
forward pass, regardless of parameter count (measured directly in `02.03`).

### The local-gradient recipe

Reverse mode needs one thing per operation: its **local gradient**. Multiplication $z=ab$ has
local gradients $\partial z/\partial a=b$, $\partial z/\partial b=a$; addition $z=a+b$ has
local gradients $1,1$. Backprop then multiplies the incoming "upstream gradient" by each local
gradient and routes the result backward along every edge, *summing* at fan-out nodes. Chain
enough of these together and you can differentiate an arbitrarily deep expression without ever
writing its derivative symbolically — exactly what we build below.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Drawing $z = (x + y) \cdot x$ as a DAG makes the fan-out case (where $x$ feeds two downstream
nodes) visible: the forward pass flows left-to-right computing values, and the backward pass
(arrows reversed) flows right-to-left accumulating gradients, summing wherever a node forked.

In [ ]:
# Draw the computation graph for z=(x+y)*x, with forward values (nodes) and local
# gradients (edges) -- via networkx/matplotlib, no external 'dot' binary needed.
x_val, y_val = 3.0, 4.0
a_val = x_val + y_val
z_val = a_val * x_val
dz_da, dz_dx_direct = x_val, a_val          # local gradients of z=a*x
dz_dx_via_add = dz_da * 1.0                  # chained through a=x+y
dz_dx_total = dz_dx_direct + dz_dx_via_add   # FAN-OUT: sum the two paths x contributed through

G = nx.DiGraph()
G.add_edges_from([("x", "a"), ("y", "a"), ("a", "z"), ("x", "z")])
pos = {"x": (0, 0), "y": (0, 2), "a": (1.5, 1), "z": (3, 1)}
fig, ax = plt.subplots(figsize=(9, 5))
nx.draw_networkx_nodes(G, pos, node_size=1800, node_color="#4C72B0", alpha=0.85, ax=ax)
nx.draw_networkx_edges(G, pos, arrowsize=20, node_size=1800, ax=ax)
labels = {"x": f"x={x_val}", "y": f"y={y_val}", "a": f"a=x+y\n={a_val}", "z": f"z=a*x\n={z_val}"}
nx.draw_networkx_labels(G, pos, labels, font_color="white", font_size=9, ax=ax)
edge_grads = {("x", "a"): "1", ("y", "a"): "1", ("a", "z"): f"{x_val}", ("x", "z"): f"{a_val}"}
for (u, v), label in edge_grads.items():
    mx, my = (pos[u][0] + pos[v][0]) / 2, (pos[u][1] + pos[v][1]) / 2
    ax.annotate(f"local grad={label}", (mx, my), fontsize=8, color="#C44E52", ha="center")
ax.set_title("Forward values (nodes) + local gradients (edges) for z=(x+y)*x")
ax.axis("off")
plt.tight_layout()
plt.show()

print(f"Total dz/dx = (direct path) {dz_dx_direct} + (via a) {dz_dx_via_add} = {dz_dx_total}")
print(f"Analytic check: z=(x+y)x=x^2+xy, so dz/dx=2x+y = {2*x_val + y_val}")

## 🐍 Implementation from Scratch

We build a minimal reverse-mode autodiff engine: a `Value` class that records every operation
it's used in, then a `backward()` method that walks the recorded graph in reverse, applying
the local-gradient recipe from the Theory section at every step. This is a miniature version of
exactly what `torch.autograd` does internally.

In [ ]:
class Value:
    """A scalar that remembers how it was computed, so gradients can flow backward through it --
    the smallest possible version of what torch.Tensor does under the hood."""
    def __init__(self, data, _parents=(), _local_grads=()):
        self.data = data
        self.grad = 0.0
        self._parents = _parents          # Value objects that produced this one
        self._local_grads = _local_grads  # d(self)/d(parent), for each parent, in order

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data + other.data, (self, other), (1.0, 1.0))  # d(a+b)=1,1

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data * other.data, (self, other), (other.data, self.data))  # d(ab)=b,a

    def tanh(self):
        t = np.tanh(self.data)
        return Value(t, (self,), (1 - t**2,))  # d(tanh(a))/da = 1 - tanh(a)^2

    def backward(self):
        order, visited = [], set()
        def build(v):  # topological sort: process a node only after all its consumers
            if id(v) not in visited:
                visited.add(id(v))
                for p in v._parents:
                    build(p)
                order.append(v)
        build(self)
        self.grad = 1.0  # dz/dz = 1, the seed of every reverse-mode pass
        for v in reversed(order):
            for parent, local_grad in zip(v._parents, v._local_grads):
                parent.grad += v.grad * local_grad  # chain rule; += handles fan-out

# Reproduce z = (x+y)*x from the Visual Intuition cell
x, y = Value(3.0), Value(4.0)
z = (x + y) * x
z.backward()
print(f"z.data={z.data} (expect 21), x.grad={x.grad} (expect 2x+y=10), y.grad={y.grad} (expect x=3)")

# A deeper graph with a nonlinearity: a tiny 1-neuron "network"
w1, w2, b = Value(0.5), Value(-0.3), Value(0.1)
out = (w1 * Value(2.0) + w2 * Value(1.0) + b).tanh()
out.backward()
print(f"\nNeuron output = tanh(w1*x1+w2*x2+b) = {out.data:.5f}")
print(f"d(out)/d(w1)={w1.grad:.5f}, d(out)/d(w2)={w2.grad:.5f}, d(out)/d(b)={b.grad:.5f}")

# Cross-check against PyTorch's autograd -- a correct from-scratch engine must match exactly
import torch
w1_t, w2_t, b_t = (torch.tensor(v, requires_grad=True) for v in (0.5, -0.3, 0.1))
torch.tanh(w1_t * 2.0 + w2_t * 1.0 + b_t).backward()
print(f"PyTorch cross-check -- d(out)/d(w1): {w1_t.grad.item():.5f} (ours: {w1.grad:.5f})")

## 🤖 Why This Matters for AI

`torch.autograd`, `tf.GradientTape`, and JAX's `grad` are all reverse-mode automatic
differentiation engines — larger and vectorized, but built from exactly the `Value` class
above: every tensor op records its inputs and local gradient rule, and `.backward()` walks the
graph in reverse. **Backpropagation is not separate from the chain rule** — it *is* the
multivariate chain rule via reverse-mode differentiation over the DAG a forward pass traces
out. Every weight is a leaf feeding the scalar loss at the root, and `.backward()` computes
every leaf's gradient in one traversal. Below, a tiny 2-layer network built entirely from our
`Value` class (no `torch.nn`) learns a toy task with plain gradient descent.

In [ ]:
# Training a tiny 2-layer network using ONLY our from-scratch Value/backward() engine --
# no torch.nn, no autograd library -- to prove reverse-mode chain rule is all learning needs.
rng = np.random.default_rng(0)

class Neuron:
    def __init__(self, n_in):
        self.w = [Value(rng.normal(scale=0.5)) for _ in range(n_in)]
        self.b = Value(0.0)
    def __call__(self, xs):
        act = self.b
        for wi, xi in zip(self.w, xs):
            act = act + wi * xi
        return act.tanh()
    def parameters(self):
        return self.w + [self.b]

class MLP:
    def __init__(self, n_in, n_hidden):
        self.hidden = [Neuron(n_in) for _ in range(n_hidden)]
        self.output = Neuron(n_hidden)
    def __call__(self, xs):
        return self.output([neuron(xs) for neuron in self.hidden])
    def parameters(self):
        return [p for n in self.hidden for p in n.parameters()] + self.output.parameters()

# Toy task: learn an XOR-like boundary -- impossible for a single linear neuron, which is
# exactly why the chain rule through a HIDDEN layer matters
X_toy = [(0.0, 0.0), (0.0, 1.0), (1.0, 0.0), (1.0, 1.0)]
y_toy = [0.0, 1.0, 1.0, 0.0]
net = MLP(n_in=2, n_hidden=4)
losses, lr = [], 0.3
for step in range(200):
    total_loss = Value(0.0)
    for xi, yi in zip(X_toy, y_toy):
        diff = net([Value(xi[0]), Value(xi[1])]) + Value(-yi)
        total_loss = total_loss + diff * diff  # squared error, via our own __add__/__mul__
    for p in net.parameters():
        p.grad = 0.0            # reset gradients, same as optimizer.zero_grad()
    total_loss.backward()        # ONE call computes gradients for every parameter in the net
    for p in net.parameters():
        p.data -= lr * p.grad    # plain gradient descent -- no momentum/Adam needed here
    losses.append(total_loss.data)

print(f"Loss: {losses[0]:.4f} -> {losses[-1]:.4f} (learned via chain rule alone, zero library autograd)")
for xi, yi in zip(X_toy, y_toy):
    pred = net([Value(xi[0]), Value(xi[1])])
    print(f"  input={xi}  target={yi}  predicted={pred.data:.3f}")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(losses, color="#4C72B0", lw=2)
ax.set_xlabel("training step"); ax.set_ylabel("total squared error")
ax.set_title("A network trained with our from-scratch Value.backward() -- pure chain rule")
plt.show()

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. By hand, apply the multivariate chain rule to $z=(x+y)\cdot(x-y)$ to find $\partial z/\partial x$
   and $\partial z/\partial y$ at $(x,y)=(3,1)$. Verify with `Value` and `.backward()`.

<details>
<summary>💡 Solution</summary>

Let $a=x+y$, $b=x-y$, $z=ab$. By the product rule (or the sum-over-paths rule, since both $a$
and $b$ depend on $x$ and on $y$): $\partial z/\partial x = \frac{\partial z}{\partial a}
\frac{\partial a}{\partial x} + \frac{\partial z}{\partial b}\frac{\partial b}{\partial x} =
b(1) + a(1) = a+b = 2x$, and similarly $\partial z/\partial y = b(1)+a(-1) = b-a = -2y$. At
$(3,1)$: $\partial z/\partial x = 2(3)=6$, $\partial z/\partial y=-2(1)=-2$ (matching the direct
expansion $z=x^2-y^2$, whose partials are trivially $2x,-2y$). Building
`z = (Value(3.0)+Value(1.0)) * (Value(3.0)-Value(1.0))` — note `Value` needs a `__sub__` or
equivalent `a + b*(-1)` construction, since it isn't defined in the notebook's own listing —
and calling `.backward()` reproduces `x.grad=6.0`, `y.grad=-2.0` exactly.

</details>

### 💭 UNDERSTAND
2. A learner differentiates $y=\sin(x^2+1)$ and writes: "by the chain rule,
   $dy/dx = \cos(x)\cdot 2x$" — reasoning that $f(u)=\sin(u)$ has derivative $\cos(u)$, so they
   plug in $\cos(x)$ directly. Identify precisely what's wrong with this, compute the correct
   derivative, and show the two disagree numerically at $x=1.5$.

<details>
<summary>💡 Solution</summary>

The error is evaluating the outer function's derivative at the *wrong* point: the chain rule
$\frac{dy}{dx}=f'(g(x))\cdot g'(x)$ requires $f'$ evaluated at $g(x)$ (the *inner* function's
output), not at $x$ itself. Here $g(x)=x^2+1$, $f(u)=\sin(u)$ so $f'(u)=\cos(u)$, and the
correct derivative is $\cos(x^2+1)\cdot 2x$ — not $\cos(x)\cdot 2x$. At $x=1.5$: the correct
value is $2(1.5)\cos(1.5^2+1) = 3\cos(3.25)\approx-2.982$ (matching a finite-difference check
of $y=\sin(x^2+1)$ directly), while the learner's version gives
$2(1.5)\cos(1.5)\approx0.212$ — not just a slightly-off number but the wrong *sign*. This is
precisely why the computation-graph picture matters: $g(x)$'s *value* has to be cached during
the forward pass and fed into $f'$ during the backward pass — a habit `Value.backward()`
enforces automatically by storing `self.data` on every node, which is exactly the bookkeeping a
by-hand chain-rule calculation has to replicate manually and is easy to skip.

</details>

### ✏️ DERIVE
3. For $z = xy + x^2$, $x$ fans out into two nodes (a multiplication with $y$, and a squaring).
   Using the sum-over-paths formula from the Theory section, derive $\partial z/\partial x$ by
   separately computing each path's contribution and summing them, then confirm your result
   matches differentiating $z$ directly as a single expression in $x$ (holding $y$ fixed).

<details>
<summary>💡 Solution outline</summary>

Let $u_1=xy$ and $u_2=x^2$, so $z=u_1+u_2$. Path 1: $\frac{\partial z}{\partial u_1}=1$,
$\frac{\partial u_1}{\partial x}=y$, contributing $1\cdot y=y$. Path 2:
$\frac{\partial z}{\partial u_2}=1$, $\frac{\partial u_2}{\partial x}=2x$, contributing
$1\cdot2x=2x$. Summing over both paths (the fan-out rule):
$$\frac{\partial z}{\partial x} = y + 2x.$$
Differentiating $z=xy+x^2$ directly with $y$ held fixed gives the identical
$\partial z/\partial x = y+2x$ by the ordinary sum and power rules — confirming the
sum-over-paths formula isn't a different rule, it's the multivariate chain rule specialized to
exactly recover single-variable differentiation once you add up every route $x$'s influence
takes to reach $z$. Numerically at $(x,y)=(3,4)$: $y+2x=4+6=10$, matching both a direct
`Value`-engine computation of $z=(x*y)+(x*x)$ and a finite-difference check.

</details>

### 🐍 IMPLEMENT
4. Add a `__pow__` method to `Value` supporting `v ** n` for a constant integer $n$, with local
   gradient $n\cdot v^{n-1}$ (power rule, `02.02`). Test on $f(x)=x^3$ at $x=2$, and check
   `x.grad` against both the closed form $3x^2$ and PyTorch's autograd on the same computation.

<details>
<summary>✅ How to know you're right</summary>

`x.grad` should equal exactly $3(2)^2=12$ after calling `(x**3).backward()` with `x=Value(2.0)`
— and `torch.tensor(2.0, requires_grad=True) ** 3`, backpropagated, should report the identical
`12.0`. As a broader check (not just this one input), verify on 3–4 other values of $x$ and $n$
too, since a `__pow__` that happens to work only at $x=2,n=3$ could still have an off-by-one
error in the exponent (a common mistake: writing the local gradient as $n\cdot v^n$ or
$v^{n-1}$ without the factor of $n$, both of which would only be caught by testing a value
where they don't coincidentally match).

</details>

### 🤖 APPLY
5. Build a small weight-sharing scenario: one shared `Value` weight `w` is multiplied by two
   *different* scalar inputs `x1`, `x2` (through `tanh`), producing `branch1`, `branch2`, which
   feed into a single scalar loss `loss = (branch1 - branch2) ** 2`. Call `.backward()` once
   and inspect `w.grad`. Then rebuild the same computation with **two separate** `Value` objects
   (same initial data, but not the same object) standing in for `w` in each branch, and compare.

<details>
<summary>✅ What you should observe</summary>

With `w` genuinely shared, `w.grad` after `.backward()` should reflect *both* branches' pull on
it simultaneously — in one concrete run, $w.\text{grad}\approx3.623$ — matching a direct
sum-over-paths hand calculation exactly. With two separate (unshared) `Value` objects instead,
each one's `.grad` reflects only *its own* branch: the first object's gradient
($\approx1.672$ in the same run) matches only the shared version's "branch-1-only"
contribution, not the full combined gradient. This is a direct, minimal illustration of why
weight sharing (embeddings reused between a transformer's input and output projection, or
convolution kernels reused at every spatial location) is handled correctly automatically —
whenever the *same* node object is reused, `Value`'s `+=` accumulation in `backward()` naturally
sums every path's contribution, with no special-case code required anywhere for the sharing
itself.

</details>

### 🚀 CHALLENGE
6. `Value.backward()`'s topological sort (`build`) is recursive — one Python function call per
   node, nested as deep as the graph is long. Construct $x^N$ as a chain of $N$ repeated
   `Value` multiplications (`result = result * x`, `N` times) for increasing $N$, and find the
   point at which `.backward()` starts raising `RecursionError`. Then investigate a fix:
   rewrite `build`'s traversal to be iterative (an explicit stack instead of recursive calls),
   and confirm the same deep chain that used to crash now runs and gives the correct gradient.

<details>
<summary>🔍 What a strong answer covers</summary>

A strong answer reports the exact crossover: with Python's default recursion limit of $1000$,
chains up to around $N\approx995$ succeed, and $N=998$ and beyond raise `RecursionError:
maximum recursion depth exceeded` — because `build` recurses once per node in the chain, and
`backward`'s own `build(self)` call adds a few more frames on top of whatever the interpreter
was already using, using up the budget before $N$ reaches exactly $1000$. Rewriting `build` to
use an explicit stack (pushing nodes to visit and popping them in a loop, rather than Python
function-call recursion) removes the limit entirely — the same test that crashed at $N=998$
should now succeed cleanly for $N$ in the thousands or tens of thousands, still producing the
exact same gradient as the closed form $N x^{N-1}$ predicts. A strong answer connects this to
why real frameworks don't hit this wall in practice: `torch.autograd`'s graph traversal is
implemented iteratively (in C++, with an explicit worklist) rather than via naive per-node
Python recursion — a computation graph with real depth (a 1000+-layer network, or a long
recurrent unroll) is exactly the regime where a recursive toy implementation's hidden
implementation-detail limit (the interpreter's call stack) would otherwise silently cap how
deep a model can be trained at all, unrelated to the actual mathematics of the chain rule.

</details>

---

## 📚 Further Reading
- Karpathy, [micrograd](https://github.com/karpathy/micrograd) — direct inspiration for this notebook's `Value` engine
- Baydin et al., ["Automatic Differentiation in Machine Learning: a Survey"](https://arxiv.org/abs/1502.05767)
- Rumelhart, Hinton & Williams, ["Learning representations by back-propagating errors"](https://www.nature.com/articles/323533a0) (1986)

---

*Next notebook: [Gradients and Directional Derivatives](05_gradients_and_directional_derivatives.ipynb)*